# Open Shots vs Wins Analysis (2024-25 Season)

This notebook charts two relationships:
1. **Actual Wins vs Estimated Open Shots** (from our model's `contest_label`)
2. **Actual Wins vs Actual Open Shots** (from NBA.com closest defender data)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 2024-25 Season Wins (from Basketball-Reference)
wins_data = {
    'CLE': 64, 'BOS': 61, 'NYK': 51, 'IND': 50, 'MIL': 48,
    'DET': 44, 'ORL': 41, 'ATL': 40, 'CHI': 39, 'MIA': 37,
    'TOR': 30, 'BKN': 26, 'PHI': 24, 'CHA': 19, 'WAS': 18,
    'OKC': 68, 'HOU': 52, 'LAL': 50, 'DEN': 50, 'LAC': 50,
    'MIN': 49, 'GSW': 48, 'MEM': 48, 'SAC': 40, 'DAL': 39,
    'PHX': 36, 'POR': 36, 'SAS': 34, 'NOP': 21, 'UTA': 17
}

wins_df = pd.DataFrame([
    {'teamTricode': team, 'wins': wins}
    for team, wins in wins_data.items()
])

print(f"Loaded wins for {len(wins_df)} teams")
print(f"Win range: {wins_df['wins'].min()} - {wins_df['wins'].max()}")

## 1. Estimated Open Shots (from our model)

In [ ]:
# Load enriched shot data for 2024-25 season
shots = pd.read_csv('enriched_data/nbastatsv3_2024_enriched_shots.csv')
print(f"Loaded {len(shots):,} shots")

# Filter to regular season only (gameId starts with '002')
shots['gameId'] = shots['gameId'].astype(str).str.zfill(10)
shots = shots[shots['gameId'].str.startswith('002')]
print(f"Regular season shots: {len(shots):,}")

# Count estimated open shots by team (contest_label == 'likely_open')
estimated_open = shots[shots['contest_label'] == 'likely_open'].groupby('teamTricode').size().reset_index(name='estimated_open_shots')
total_shots = shots.groupby('teamTricode').size().reset_index(name='total_shots')

# Merge
team_stats = total_shots.merge(estimated_open, on='teamTricode', how='left')
team_stats['estimated_open_shots'] = team_stats['estimated_open_shots'].fillna(0).astype(int)
team_stats['estimated_open_pct'] = (team_stats['estimated_open_shots'] / team_stats['total_shots'] * 100).round(1)

# Merge with wins
team_stats = team_stats.merge(wins_df, on='teamTricode', how='left')

print(f"\nTeams with estimated open shot data: {len(team_stats)}")
team_stats.sort_values('estimated_open_shots', ascending=False).head(10)

In [ ]:
# Chart 1: Actual Wins vs Estimated Open Shots
fig, ax = plt.subplots(figsize=(12, 8))

ax.scatter(team_stats['estimated_open_shots'], team_stats['wins'], s=100, alpha=0.7, c='steelblue')

# Add team labels
for _, row in team_stats.iterrows():
    ax.annotate(row['teamTricode'], (row['estimated_open_shots'], row['wins']),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

# Add trend line
z = np.polyfit(team_stats['estimated_open_shots'], team_stats['wins'], 1)
p = np.poly1d(z)
x_line = np.linspace(team_stats['estimated_open_shots'].min(), team_stats['estimated_open_shots'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', alpha=0.7, label=f'Trend (slope={z[0]:.4f})')

# Correlation
corr = team_stats['estimated_open_shots'].corr(team_stats['wins'])

ax.set_xlabel('Estimated Open Shots (contest_label = likely_open)', fontsize=12)
ax.set_ylabel('Actual Wins (2024-25)', fontsize=12)
ax.set_title(f'Actual Wins vs Estimated Open Shots\nCorrelation: r = {corr:.3f}', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Correlation between Estimated Open Shots and Wins: r = {corr:.3f}")

## 2. Actual Open Shots (from NBA.com)

**Definition**: Open shots = closest defender 4+ feet away (combining "4-6 Feet Open" + "6+ Feet Wide Open")

**Source**: https://www.nba.com/stats/teams/shots-closest-defender (2024-25 Regular Season, Totals)

In [ ]:
# NBA.com Open Shots Data (2024-25 Season)
# Combined: 4-6 Feet (Open) + 6+ Feet (Wide Open) = All shots with defender 4+ feet away
# Source: https://www.nba.com/stats/teams/shots-closest-defender

# 4-6 Feet FGA + 6+ Feet FGA = Total Open FGA
actual_open_data = {
    'ATL': 1980 + 2055,  # Atlanta Hawks
    'BOS': 2353 + 2002,  # Boston Celtics
    'BKN': 2116 + 1780,  # Brooklyn Nets
    'CHA': 2146 + 1759,  # Charlotte Hornets
    'CHI': 2058 + 2368,  # Chicago Bulls
    'CLE': 2209 + 1870,  # Cleveland Cavaliers
    'DAL': 2188 + 1436,  # Dallas Mavericks
    'DEN': 2007 + 1706,  # Denver Nuggets
    'DET': 2104 + 1475,  # Detroit Pistons
    'GSW': 2206 + 1961,  # Golden State Warriors
    'HOU': 2319 + 1699,  # Houston Rockets
    'IND': 2169 + 2204,  # Indiana Pacers
    'LAC': 2052 + 1385,  # LA Clippers
    'LAL': 2281 + 1690,  # Los Angeles Lakers
    'MEM': 2011 + 2077,  # Memphis Grizzlies
    'MIA': 2077 + 1683,  # Miami Heat
    'MIL': 2448 + 1720,  # Milwaukee Bucks
    'MIN': 2192 + 1866,  # Minnesota Timberwolves
    'NOP': 2084 + 1651,  # New Orleans Pelicans
    'NYK': 2411 + 1444,  # New York Knicks
    'OKC': 2442 + 2242,  # Oklahoma City Thunder
    'ORL': 2123 + 1797,  # Orlando Magic
    'PHI': 2247 + 1727,  # Philadelphia 76ers
    'PHX': 2149 + 1914,  # Phoenix Suns
    'POR': 2134 + 1803,  # Portland Trail Blazers
    'SAC': 2199 + 1801,  # Sacramento Kings
    'SAS': 2280 + 1868,  # San Antonio Spurs
    'TOR': 2060 + 1781,  # Toronto Raptors
    'UTA': 2285 + 1540,  # Utah Jazz
    'WAS': 2178 + 1956,  # Washington Wizards
}

actual_open_df = pd.DataFrame([
    {'teamTricode': team, 'actual_open_shots': fga}
    for team, fga in actual_open_data.items()
])

# Merge with wins
actual_open_df = actual_open_df.merge(wins_df, on='teamTricode', how='left')
print(f"Loaded actual open shot data for {len(actual_open_df)} teams")
print(f"Open shots range: {actual_open_df['actual_open_shots'].min()} - {actual_open_df['actual_open_shots'].max()}")
actual_open_df.sort_values('actual_open_shots', ascending=False).head(10)

In [ ]:
# Chart 2: Actual Wins vs Actual Open Shots (from NBA.com)
if actual_open_data:
    fig, ax = plt.subplots(figsize=(12, 8))
    
    ax.scatter(actual_open_df['actual_open_shots'], actual_open_df['wins'], s=100, alpha=0.7, c='forestgreen')
    
    # Add team labels
    for _, row in actual_open_df.iterrows():
        ax.annotate(row['teamTricode'], (row['actual_open_shots'], row['wins']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    # Add trend line
    z = np.polyfit(actual_open_df['actual_open_shots'], actual_open_df['wins'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(actual_open_df['actual_open_shots'].min(), actual_open_df['actual_open_shots'].max(), 100)
    ax.plot(x_line, p(x_line), 'r--', alpha=0.7, label=f'Trend (slope={z[0]:.4f})')
    
    # Correlation
    corr = actual_open_df['actual_open_shots'].corr(actual_open_df['wins'])
    
    ax.set_xlabel('Actual Open Shots (NBA.com: Defender 4+ ft)', fontsize=12)
    ax.set_ylabel('Actual Wins (2024-25)', fontsize=12)
    ax.set_title(f'Actual Wins vs Actual Open Shots (NBA.com)\nCorrelation: r = {corr:.3f}', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Correlation between Actual Open Shots and Wins: r = {corr:.3f}")
else:
    print("Chart 2 requires actual open shot data from NBA.com.")
    print("Please fill in the actual_open_data dictionary in the cell above.")

## 3. Comparison: Estimated vs Actual Open Shots

In [ ]:
if actual_open_data:
    # Merge estimated and actual
    comparison = team_stats[['teamTricode', 'estimated_open_shots', 'wins']].merge(
        actual_open_df[['teamTricode', 'actual_open_shots']], on='teamTricode', how='inner'
    )
    
    # Side-by-side charts
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Chart 1: Estimated
    ax1 = axes[0]
    ax1.scatter(comparison['estimated_open_shots'], comparison['wins'], s=100, alpha=0.7, c='steelblue')
    for _, row in comparison.iterrows():
        ax1.annotate(row['teamTricode'], (row['estimated_open_shots'], row['wins']),
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    corr1 = comparison['estimated_open_shots'].corr(comparison['wins'])
    ax1.set_xlabel('Estimated Open Shots', fontsize=11)
    ax1.set_ylabel('Wins', fontsize=11)
    ax1.set_title(f'Estimated Open Shots vs Wins\nr = {corr1:.3f}', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Chart 2: Actual
    ax2 = axes[1]
    ax2.scatter(comparison['actual_open_shots'], comparison['wins'], s=100, alpha=0.7, c='forestgreen')
    for _, row in comparison.iterrows():
        ax2.annotate(row['teamTricode'], (row['actual_open_shots'], row['wins']),
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    corr2 = comparison['actual_open_shots'].corr(comparison['wins'])
    ax2.set_xlabel('Actual Open Shots (NBA.com)', fontsize=11)
    ax2.set_ylabel('Wins', fontsize=11)
    ax2.set_title(f'Actual Open Shots vs Wins\nr = {corr2:.3f}', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nCorrelation Comparison:")
    print(f"  Estimated Open Shots vs Wins: r = {corr1:.3f}")
    print(f"  Actual Open Shots vs Wins:    r = {corr2:.3f}")
else:
    print("Comparison requires actual open shot data from NBA.com.")